# 📈 Notebook 02 — Visualisation de données

La visualisation est essentielle pour **comprendre**, **explorer** et **communiquer** vos données.

### 🎯 Objectifs de ce notebook
- Maîtriser **Matplotlib** pour les graphiques de base
- Utiliser **Seaborn** pour des visualisations statistiques élégantes
- Créer des graphiques interactifs avec **Plotly**
- Choisir le bon graphique selon vos données

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Style global
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Datasets
titanic  = sns.load_dataset('titanic')
tips     = sns.load_dataset('tips')
iris     = sns.load_dataset('iris')

print("✅ Librairies et datasets chargés !")

## 1. Matplotlib — La base

In [ ]:
# Graphique linéaire
np.random.seed(42)
jours = np.arange(1, 31)
ventes = 100 + np.cumsum(np.random.randn(30) * 10)

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(jours, ventes, color='steelblue', linewidth=2, marker='o', markersize=4, label='Ventes')
ax.axhline(ventes.mean(), color='red', linestyle='--', alpha=0.7, label=f'Moyenne ({ventes.mean():.0f})')
ax.fill_between(jours, ventes, ventes.mean(), alpha=0.1, color='steelblue')

ax.set_title('Évolution des ventes sur 30 jours', fontsize=14, pad=15)
ax.set_xlabel('Jour')
ax.set_ylabel('Ventes (€)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Subplots — plusieurs graphiques côte à côte
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histogramme
data = np.random.normal(0, 1, 500)
axes[0].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Histogramme')

# Barres
categories = ['A', 'B', 'C', 'D', 'E']
valeurs = [23, 45, 31, 67, 42]
axes[1].bar(categories, valeurs, color=sns.color_palette('husl', 5))
axes[1].set_title('Graphique en barres')

# Nuage de points
x = np.random.randn(100)
y = x * 0.8 + np.random.randn(100) * 0.5
axes[2].scatter(x, y, alpha=0.6, color='coral')
axes[2].set_title('Nuage de points')

plt.suptitle('Les graphiques de base avec Matplotlib', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Seaborn — Visualisation statistique

In [ ]:
# Distribution avec kde
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution de l'âge par classe
sns.histplot(data=titanic.dropna(subset=['age']), x='age', hue='pclass',
             multiple='dodge', bins=20, ax=axes[0], palette='Set2')
axes[0].set_title('Distribution de l\'âge par classe (Titanic)')

# KDE (densité)
for pclass in [1, 2, 3]:
    subset = titanic[titanic['pclass'] == pclass]['age'].dropna()
    sns.kdeplot(subset, ax=axes[1], label=f'Classe {pclass}', fill=True, alpha=0.3)
axes[1].set_title('Densité de l\'âge par classe')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot et violinplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=tips, x='day', y='total_bill', hue='sex', ax=axes[0], palette='pastel')
axes[0].set_title('Total des additions par jour (Boxplot)')
axes[0].set_xlabel('Jour')

sns.violinplot(data=tips, x='day', y='total_bill', hue='sex', ax=axes[1],
               split=True, palette='muted')
axes[1].set_title('Distribution des additions par jour (Violin)')
axes[1].set_xlabel('Jour')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(8, 6))

corr = iris.drop('species', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, center=0)

ax.set_title('Matrice de corrélation — Dataset Iris', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot — vue globale des relations
g = sns.pairplot(iris, hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.6}, height=2.5)
g.fig.suptitle('Pairplot — Dataset Iris', y=1.02)
plt.show()

## 3. Plotly — Graphiques interactifs

In [ ]:
# Scatter interactif
fig = px.scatter(
    iris,
    x='sepal_length', y='sepal_width',
    color='species', size='petal_length',
    hover_data=['petal_width'],
    title='Iris — Longueur vs Largeur du sépale (interactif)',
    labels={'sepal_length': 'Longueur sépale', 'sepal_width': 'Largeur sépale'}
)
fig.show()

In [ ]:
# Barres empilées interactives
survie = titanic.groupby(['pclass', 'survived']).size().reset_index(name='count')
survie['survived'] = survie['survived'].map({0: 'Décédé', 1: 'Survivant'})
survie['pclass'] = survie['pclass'].map({1: '1ère classe', 2: '2ème classe', 3: '3ème classe'})

fig = px.bar(
    survie,
    x='pclass', y='count', color='survived',
    barmode='group',
    title='Titanic — Survie par classe de voyage',
    labels={'pclass': 'Classe', 'count': 'Nombre de passagers', 'survived': 'Statut'},
    color_discrete_map={'Survivant': '#2ecc71', 'Décédé': '#e74c3c'}
)
fig.show()

In [ ]:
# Courbe temporelle interactive
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=12, freq='ME')
produits = ['Produit A', 'Produit B', 'Produit C']

ts_data = pd.DataFrame({
    'date': np.tile(dates, 3),
    'produit': np.repeat(produits, 12),
    'ventes': np.concatenate([
        100 + np.cumsum(np.random.randn(12) * 15),
        80  + np.cumsum(np.random.randn(12) * 10),
        120 + np.cumsum(np.random.randn(12) * 20)
    ])
})

fig = px.line(
    ts_data, x='date', y='ventes', color='produit',
    title='Évolution des ventes par produit (2024)',
    markers=True
)
fig.show()

## 4. Guide : Quel graphique choisir ?

| Objectif | Graphique recommandé |
|----------|---------------------|
| Évolution dans le temps | Courbe (`lineplot`) |
| Comparaison de catégories | Barres (`barplot`) |
| Distribution d'une variable | Histogramme, KDE (`histplot`, `kdeplot`) |
| Dispersion / outliers | Boxplot, Violinplot |
| Relation entre 2 variables | Nuage de points (`scatterplot`) |
| Corrélations | Heatmap |
| Part d'un tout | Camembert (`pie`) — *à utiliser avec modération* |
| Vue d'ensemble multivariée | Pairplot |

## 5. ✏️ Exercices pratiques

In [ ]:
# EXERCICE 1
# Avec le dataset 'tips' (sns.load_dataset('tips')) :
# - Créez un barplot du pourboire moyen par jour de la semaine
# - Ajoutez un titre et des labels d'axes

# Votre code ici


In [ ]:
# EXERCICE 2
# Avec Plotly, créez un graphique interactif de votre choix
# sur le dataset iris ou titanic

# Votre code ici


---

## ✅ Résumé du notebook 02

| Librairie | Points forts | Usage |
|-----------|-------------|-------|
| **Matplotlib** | Contrôle total, personnalisable | Graphiques précis, publication |
| **Seaborn** | Statistiques, élégant | Exploration, corrélations |
| **Plotly** | Interactif, web-ready | Dashboards, présentation |

## ➡️ Prochain notebook : [03 — Statistiques](03_statistiques.ipynb)

---
*Formation Data Science Python — Notebook 02/04*